# PPM lowering: a two-step sequence, with and without the driver

One PPM sequence on THREE patches in a row — `M(Z̄A Z̄B)` through
the corridor between A and B, then `M(Z̄B Z̄C)` through the corridor
between B and C — built two ways: through the kernel API alone
(`lower_ppm` / `apply_plan` + builder primitives), then through
`SequentialPPMExperiment`.  Each shows the full-circuit
`detslice-with-ops-svg` (every tick).


In [ ]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import contextlib
import io

import stim

import lightstim
from lightstim.ir.builder import CircuitBuilder
from lightstim.ir.qec_system import QECSystem
from lightstim.ir.tracker import SyndromeTracker
from lightstim.qec_code.surface_code.rotated import RotatedSurfaceCode
from lightstim.qec_code.surface_code.rotated.bent_joint_se import (
    se_round_chunk)
from lightstim.protocols.ppm import (PatchSpec, PPMStep, PPMRequest,
                                     SequentialPPMExperiment, apply_plan,
                                     lower_ppm, origin_of)

# the environment may alias bare lightstim imports elsewhere via a .pth -
# always verify which tree this notebook runs against
print("lightstim from:", lightstim.__file__)


In [ ]:
D = 3          # code distance (rounds = D in every merged window)

px = [PatchSpec('A', origin_of(0, 0, D, seam=True), D, 'X_horizontal'),
      PatchSpec('B', origin_of(2, 0, D, seam=True), D, 'X_horizontal'),
      PatchSpec('C', origin_of(4, 0, D, seam=True), D, 'X_horizontal')]
STATES = {'A': 'Z', 'B': 'Z', 'C': 'Z'}
# two DIFFERENT products through two different corridors
requests = [PPMRequest(targets=(('A', 'Z'), ('B', 'Z')), route=((1, 0),)),
            PPMRequest(targets=(('B', 'Z'), ('C', 'Z')), route=((3, 0),))]


## 1. Without `SequentialPPMExperiment`: the kernel API end to end


In [ ]:
system = QECSystem()
for s in px:
    p = RotatedSurfaceCode(distance=s.distance)
    if s.orientation == 'X_horizontal':
        p.transpose_coords()
    system.add_patch(p, name=s.name,
                     offset=(s.origin[0] - 1, s.origin[1] - 1))

tracker = SyndromeTracker(num_qubits=system.num_qubits,
                          expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system,
                         if_detector=True)
system.register_tracker(tracker)
system.register_builder(builder)
builder.write_coordinates()
owner = system.index_to_owner_map
builder.initialize(
    init_dict={q: STATES[owner[q]] for q in system.data_indices
               if owner.get(q) in STATES},
    n=system.num_qubits)
orient = {s.name: s.orientation for s in px}
domains = {tuple(system.qubit_coords[q]): orient[owner[q]]
           for q in system.data_indices if owner.get(q) in orient}
builder.apply_syndrome_extraction(
    circuit_chunk=se_round_chunk(system, domains=domains), rounds=1)

for i, request in enumerate(requests):
    # Define-by-run: register each coupler only after the baseline has
    # established the logical patches, immediately before that PPM.
    plan = lower_ppm(px, request, system=system)
    apply_plan(system, plan, f'ppm_{i}')
    cname = f'ppm_{i}'
    builder.activate_coupler(cname)
    cp = system.coupler_patches[cname]
    l2g = system.local_to_global_map[cname]
    corridor_init = {l2g[q]: plan.corridor_init_basis
                     for q in cp.data_indices}
    builder.initialize(init_dict=corridor_init, n=system.num_qubits)
    merged = dict(plan.route_result.layout.domains or {})
    for q in system.data_indices:
        nm = owner.get(q)
        if nm in orient:
            merged.setdefault(tuple(system.qubit_coords[q]), orient[nm])
    builder.apply_syndrome_extraction(
        circuit_chunk=se_round_chunk(system, domains=merged), rounds=D)
    builder.deactivate_coupler(cname)
    builder.apply_data_readout(final_measurements=dict(corridor_init),
                               resolve_absorbed=False)

builder.apply_data_readout(
    final_measurements={q: STATES[owner[q]] for q in system.data_indices
                        if owner.get(q) in STATES})
c_free = builder.circuit
print('qubits:', c_free.num_qubits, '| ticks:', c_free.num_ticks,
      '| detectors:', c_free.num_detectors,
      '| observables:', c_free.num_observables)
det, obs = c_free.compile_detector_sampler(seed=0).sample(
    1024, separate_observables=True)
print('p=0 silent:', not det.any() and not obs.any())
c_free.diagram('detslice-with-ops-svg', tick=range(1, c_free.num_ticks + 1))


## 2. With `SequentialPPMExperiment`: the same sequence


In [ ]:
exp = SequentialPPMExperiment(
    px,
    [PPMStep([('A', 'Z'), ('B', 'Z')], route=[(1, 0)]),
     PPMStep([('B', 'Z'), ('C', 'Z')], route=[(3, 0)])],
    initial_states=STATES, final_measure_states=STATES,
    rounds=D, rounds_init=1)
with contextlib.redirect_stdout(io.StringIO()):
    circuit = exp.build()
print('qubits:', circuit.num_qubits, '| ticks:', circuit.num_ticks,
      '| detectors:', circuit.num_detectors,
      '| observables:', circuit.num_observables)
det, obs = circuit.compile_detector_sampler(seed=0).sample(
    1024, separate_observables=True)
print('p=0 silent:', not det.any() and not obs.any())
circuit.diagram('detslice-with-ops-svg', tick=range(1, circuit.num_ticks + 1))
